# Hard Case: Elemen di dalam `<table>`

Kalau datanya ada dalam **tabel HTML** (`<table>`), tidak perlu cape-cape pakai
BeautifulSoup baris demi baris. Ada **one-liner ajaib**: `pandas.read_html()`.

`pd.read_html(...)` membaca **semua** `<table>` di halaman dan mengembalikan **list of
DataFrame** — langsung rapi, tinggal pilih tabel yang diinginkan.

> Butuh parser HTML: `lxml` atau `html5lib` (di project ini `lxml` sudah ada).

**Tooling:** `pandas` (+ `lxml`). BeautifulSoup dipakai hanya sebagai pembanding.


## 1. Baca tabel dari HTML

Kita pakai HTML berisi **dua** tabel (produk & kurs) agar terlihat bahwa `read_html`
mengembalikan **list** — satu DataFrame per tabel.

> Catatan pandas 3.x: bungkus string HTML dengan `io.StringIO(...)` saat dipakai langsung.


In [1]:
from io import StringIO

import pandas as pd

HTML_TABEL = """
<h3>Daftar Produk</h3>
<table>
  <thead><tr><th>Nama</th><th>Harga</th><th>Stok</th></tr></thead>
  <tbody>
    <tr><td>Belajar Python</td><td>Rp75.000</td><td>Tersedia</td></tr>
    <tr><td>Mouse Wireless</td><td>Rp150.000</td><td>Tersedia</td></tr>
    <tr><td>Keyboard Mekanik</td><td>Rp350.000</td><td>Habis</td></tr>
  </tbody>
</table>

<h3>Kurs Mata Uang</h3>
<table>
  <thead><tr><th>Mata Uang</th><th>Nilai (Rp)</th></tr></thead>
  <tbody>
    <tr><td>USD</td><td>16.000</td></tr>
    <tr><td>EUR</td><td>17.500</td></tr>
  </tbody>
</table>
"""

# read_html -> list of DataFrame (satu per <table>)
# flavor="lxml" supaya pakai parser lxml (tidak butuh html5lib)
tabel = pd.read_html(StringIO(HTML_TABEL), flavor="lxml")
print("Jumlah tabel ditemukan:", len(tabel))
tabel[0]


Jumlah tabel ditemukan: 2


,Nama,Harga,Stok
0,Belajar Python,Rp75.000,Tersedia
1,Mouse Wireless,Rp150.000,Tersedia
2,Keyboard Mekanik,Rp350.000,Habis


## 2. Pilih tabel dengan `match=` lalu bersihkan kolom

Kalau ada banyak tabel, pakai `match="kata kunci"` untuk memilih tabel yang **mengandung
teks tertentu**. Setelah dapat DataFrame, kolom dibersihkan seperti biasa dengan pandas.

**Membaca dari URL nyata** (mis. Wikipedia):

```python
# cara paling singkat (kalau situs mengizinkan):
tabel = pd.read_html("https://en.wikipedia.org/wiki/List_of_countries_by_population", match="Population")

# kalau situs butuh header/diblok, ambil dulu dengan requests lalu StringIO:
import requests
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10).text
tabel = pd.read_html(StringIO(html), match="Population")
```


In [2]:
# pilih tabel yang mengandung kata "Harga" (tabel produk)
produk = pd.read_html(StringIO(HTML_TABEL), match="Harga", flavor="lxml")[0]
print("Sebelum cleaning:")
print(produk, "\n")

# bersihkan kolom Harga: "Rp75.000" -> 75000 (int). Titik = pemisah ribuan.
produk["Harga"] = produk["Harga"].str.replace(r"[^0-9]", "", regex=True).astype(int)

print("Sesudah cleaning:")
print(produk)
print("\nTotal nilai stok (jumlah harga):", produk["Harga"].sum())


Sebelum cleaning:
               Nama      Harga      Stok
0    Belajar Python   Rp75.000  Tersedia
1    Mouse Wireless  Rp150.000  Tersedia
2  Keyboard Mekanik  Rp350.000     Habis 

Sesudah cleaning:
               Nama   Harga      Stok
0    Belajar Python   75000  Tersedia
1    Mouse Wireless  150000  Tersedia
2  Keyboard Mekanik  350000     Habis

Total nilai stok (jumlah harga): 575000


## 3. Pembanding: parsing tabel manual dengan BeautifulSoup

Untuk membandingkan, ini cara manual dengan BeautifulSoup. Hasilnya sama, tapi kodenya
jauh lebih panjang. Untuk tabel, **`pd.read_html` hampir selalu lebih baik**.

Manual BS4 baru menang kalau strukturnya "aneh" (bukan `<table>` standar, atau butuh
mengambil atribut seperti `href` di dalam sel).

## Latihan
Ambil tabel **kurs** (pakai `match="Mata Uang"` atau index `[1]`), lalu ubah kolom nilai
jadi angka dan hitung rata-ratanya.

> Catatan: `match` mencari teks **di dalam** `<table>`. Kata "Kurs" ada di `<h3>` (di luar
> tabel), jadi tidak bisa dipakai untuk `match` — gunakan teks header kolom.


In [3]:
from bs4 import BeautifulSoup

# --- Pembanding: BS4 manual ---
soup = BeautifulSoup(HTML_TABEL, "html.parser")
tabel_produk = soup.find("table")
baris = []
for tr in tabel_produk.find("tbody").find_all("tr"):
    baris.append([td.get_text(strip=True) for td in tr.find_all("td")])
print("Hasil parsing manual BS4:")
for b in baris:
    print("  ", b)

# --- Contoh jawaban latihan: tabel kurs ---
# match mencari teks DI DALAM <table>; "Kurs" hanya ada di <h3> (di luar tabel),
# jadi kita match pada header kolom yang ada di dalam tabel: "Mata Uang".
# thousands="." -> "16.000" dibaca sebagai 16000 (titik = pemisah ribuan, bukan desimal).
kurs = pd.read_html(StringIO(HTML_TABEL), match="Mata Uang", flavor="lxml", thousands=".")[0]
print("\nTabel kurs:")
print(kurs)
print("Tipe kolom Nilai:", kurs["Nilai (Rp)"].dtype)
print("Rata-rata nilai:", kurs["Nilai (Rp)"].mean())


Hasil parsing manual BS4:
   ['Belajar Python', 'Rp75.000', 'Tersedia']
   ['Mouse Wireless', 'Rp150.000', 'Tersedia']
   ['Keyboard Mekanik', 'Rp350.000', 'Habis']

Tabel kurs:
  Mata Uang  Nilai (Rp)
0       USD       16000
1       EUR       17500
Tipe kolom Nilai: int64
Rata-rata nilai: 16750.0
